# M5 Forecasting Accuracy — Pipeline

| ステップ | 内容 | 状態 |
|----------|------|------|
| 1. Setup | 環境・データ読み込み | ✅ |
| 2. Data Prep | wide → long 変換、型最適化 | 🔲 |
| 3. Feature Engineering | ラグ・ローリング・カレンダー・価格 | 🔲 |
| 4. Train/Val Split | d_1886〜d_1913 を validation | 🔲 |
| 5. Model Training | TBD | 🔲 |
| 6. Evaluation | WRMSSE | 🔲 |
| 7. Submission | validation + evaluation | 🔲 |

In [ ]:
# ============================================================
# SETUP — 環境検出 + parquet 確認 (CSV 一括読み込みなし)
# ============================================================
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 5)

from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

IS_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ
print(f"Environment: {'Google Colab' if IS_COLAB else 'Local'}")
print(f"CWD: {os.getcwd()}")

COMPETITION = 'm5-forecasting-accuracy'

# ============================================================
# DATA_DIR 検出 (df_features.parquet の存在で判定)
# ============================================================
DATA_DIR = None

_candidates = []

if IS_COLAB:
    _candidates = [
        Path(f'/content/{COMPETITION}'),
        Path('/content/data'),
    ]
    # Drive マウント
    _drive_ok = Path('/content/drive/MyDrive').exists()
    if not _drive_ok:
        try:
            from google.colab import drive
            drive.mount('/content/drive', force_remount=False)
            _drive_ok = True
        except Exception:
            pass
    if _drive_ok:
        _candidates += [
            Path(f'/content/drive/MyDrive/kaggle/{COMPETITION}'),
            Path(f'/content/drive/MyDrive/{COMPETITION}'),
        ]
else:
    # ノートブック自身のディレクトリ (VSCode は __vsc_ipynb_file__ を設定)
    _nb_dir = None
    if '__vsc_ipynb_file__' in globals():
        _nb_dir = Path(__vsc_ipynb_file__).parent
    _candidates = [
        Path(f'/mnt/c/Users/hiroyuki_nakayama/src/kaggle-with-mcp/{COMPETITION}'),
    ]
    if _nb_dir is not None:
        _candidates.insert(0, _nb_dir)
    _candidates += [
        Path(f'./{COMPETITION}'),
        Path('.'),
    ]

for c in _candidates:
    print(f'  checking: {c.resolve()}')
    if (c / 'df_features.parquet').exists():
        DATA_DIR = c
        break

assert DATA_DIR is not None, \
    'df_features.parquet が見つかりません。preprocess.py を先に実行してください'

DATA_DIR = str(DATA_DIR) + '/'
CACHE_PATH = DATA_DIR + 'df_features.parquet'

pf_info = pq.ParquetFile(CACHE_PATH).metadata
print(f'DATA_DIR    : {DATA_DIR}')
print(f'Parquet     : {os.path.getsize(CACHE_PATH)/1e6:.0f} MB')
print(f'  rows      : {pf_info.num_rows:,}')
print(f'  row_groups: {pf_info.num_row_groups}')
print(f'  columns   : {pf_info.num_columns}')
del pf_info

## Step 1: 定数・設定

In [ ]:
# ============================================================
# 定数・設定 (CSV を読まずに parquet schema から取得)
# ============================================================
PRED_HORIZON   = 28
N_DAYS         = 1941
VAL_START_DAY  = 1886
EVAL_START_DAY = 1914
KEEP_FROM_DAY  = 1100

META_COLS = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']

# Feature list from parquet schema
_pf = pq.ParquetFile(CACHE_PATH)
_all_cols = [f.name for f in _pf.schema_arrow]
DROP_COLS = META_COLS + ['d', 'date', 'sales', 'wm_yr_wk', 'd_num',
                         'snap_CA', 'snap_TX', 'snap_WI']
FEATURES = [c for c in _all_cols if c not in DROP_COLS]
TARGET   = 'sales'
del _pf

print(f'Features ({len(FEATURES)}):')
for i, f in enumerate(FEATURES):
    print(f'  {i+1:2d}. {f}')

## Step 2: データ分割 (parquet → memmap + val/eval)

In [ ]:
# ============================================================
# Step 2: 前処理済みデータ確認 (preprocess.py で生成済み)
# ============================================================
TRAIN_X_PATH   = DATA_DIR + 'train_X.dat'
TRAIN_Y_PATH   = DATA_DIR + 'train_y.dat'
TRAIN_CAT_PATH = DATA_DIR + 'train_cat.dat'
VAL_PATH       = DATA_DIR + 'val.parquet'
EVAL_PATH      = DATA_DIR + 'eval.parquet'

_required = {
    'train_X.dat':   TRAIN_X_PATH,
    'train_y.dat':   TRAIN_Y_PATH,
    'train_cat.dat': TRAIN_CAT_PATH,
    'val.parquet':   VAL_PATH,
    'eval.parquet':  EVAL_PATH,
}
_missing = [k for k, v in _required.items() if not os.path.exists(v)]
if _missing:
    raise FileNotFoundError(
        'ファイル不足: ' + str(_missing) + '\n'
        'preprocess.py を先に実行してください'
    )

n_train = os.path.getsize(TRAIN_X_PATH) // (len(FEATURES) * 4)
n_val   = pq.ParquetFile(VAL_PATH).metadata.num_rows
n_eval  = pq.ParquetFile(EVAL_PATH).metadata.num_rows

# cat_id マッピング (カテゴリコード → 名前)
CAT_NAMES = {0: 'FOODS', 1: 'HOBBIES', 2: 'HOUSEHOLD'}

print(f'train : {n_train:,} rows  ({os.path.getsize(TRAIN_X_PATH)/1e6:.0f} MB memmap)')
print(f'val   : {n_val:,} rows  ({os.path.getsize(VAL_PATH)/1e6:.0f} MB)')
print(f'eval  : {n_eval:,} rows  ({os.path.getsize(EVAL_PATH)/1e6:.0f} MB)')
print(f'cat_id: {os.path.getsize(TRAIN_CAT_PATH)/1e6:.0f} MB')

## Step 3: 準備完了 確認

In [ ]:
# ============================================================
# Step 3: 準備完了 確認
# ============================================================
print(f'Train : {n_train:,} rows × {len(FEATURES)} features (memmap)')
print(f'Val   : {n_val:,} rows')
print(f'Eval  : {n_eval:,} rows')
print(f'Target: {TARGET}')
print(f'\nMemmap files:')
print(f'  X: {TRAIN_X_PATH}  ({os.path.getsize(TRAIN_X_PATH)/1e6:.0f} MB)')
print(f'  y: {TRAIN_Y_PATH}  ({os.path.getsize(TRAIN_Y_PATH)/1e6:.0f} MB)')

## Step 4: Sanity Check

In [ ]:
# ============================================================
# Step 4: Sanity Check (memmap の先頭を確認)
# ============================================================
X_peek = np.memmap(TRAIN_X_PATH, dtype='float32', mode='r',
                   shape=(n_train, len(FEATURES)))
y_peek = np.memmap(TRAIN_Y_PATH, dtype='float32', mode='r',
                   shape=(n_train,))
print('X_train[:3]:')
print(pd.DataFrame(X_peek[:3], columns=FEATURES))
print(f'\ny_train[:10]: {y_peek[:10]}')
print(f'y mean: {y_peek.mean():.3f}, std: {y_peek.std():.3f}, zeros: {(y_peek==0).sum():,}/{n_train:,}')
del X_peek, y_peek

## Step 5: モデル学習

In [ ]:
# ============================================================
# Step 5: LightGBM 学習 — cat_id 別 3モデル
# ============================================================
import lightgbm as lgb

# Train: memmap
X_train = np.memmap(TRAIN_X_PATH, dtype='float32', mode='r',
                    shape=(n_train, len(FEATURES)))
y_train = np.memmap(TRAIN_Y_PATH, dtype='float32', mode='r',
                    shape=(n_train,))
cat_train = np.memmap(TRAIN_CAT_PATH, dtype='int8', mode='r',
                      shape=(n_train,))

# Val: row_group ごとに読んで concat (小さいので許容)
pf_val = pq.ParquetFile(VAL_PATH)
val_parts = []
for i in range(pf_val.metadata.num_row_groups):
    val_parts.append(pf_val.read_row_group(i).to_pandas())
df_val = pd.concat(val_parts, ignore_index=True)
del val_parts, pf_val
df_val[FEATURES] = df_val[FEATURES].fillna(0)

lgb_params = {
    'objective'              : 'tweedie',
    'tweedie_variance_power' : 1.1,
    'metric'                 : 'rmse',
    'num_leaves'             : 127,
    'learning_rate'          : 0.03,
    'feature_fraction'       : 0.8,
    'bagging_fraction'       : 0.8,
    'bagging_freq'           : 1,
    'min_child_samples'      : 20,
    'n_jobs'                 : -1,
    'seed'                   : 42,
    'verbosity'              : -1,
    'two_round'              : True,
    'max_bin'                : 127,
}

# cat_id 別に 3 モデルを学習
models = {}
for cat_id, cat_name in CAT_NAMES.items():
    print(f'\n{"="*60}')
    print(f'Training: {cat_name} (cat_id={cat_id})')
    print(f'{"="*60}')

    # Train mask
    tr_mask = (cat_train == cat_id)
    n_cat = int(tr_mask.sum())
    # memmap からインデックスで抽出
    tr_idx = np.where(tr_mask)[0]
    X_cat = X_train[tr_idx]
    y_cat = y_train[tr_idx]

    # Val mask
    val_mask = (df_val['cat_id'] == cat_id)
    df_val_cat = df_val[val_mask]

    print(f'  train: {n_cat:,} rows, val: {len(df_val_cat):,} rows')

    dtrain = lgb.Dataset(
        X_cat, label=y_cat,
        feature_name=FEATURES,
        free_raw_data=True,
    )
    dval = lgb.Dataset(
        df_val_cat[FEATURES].values, label=df_val_cat[TARGET].values,
        reference=dtrain,
        free_raw_data=True,
    )

    model = lgb.train(
        lgb_params, dtrain,
        num_boost_round=2000,
        valid_sets=[dtrain, dval],
        valid_names=['train', 'val'],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
    )
    models[cat_id] = model
    print(f'  Best iteration: {model.best_iteration}')
    print(f'  Best val RMSE : {model.best_score["val"]["rmse"]:.4f}')

    del X_cat, y_cat, tr_idx, dtrain, dval, df_val_cat

del X_train, y_train, cat_train
print(f'\n{len(models)} モデル学習完了')

## Step 6: 評価 (WRMSSE)

In [ ]:
# ============================================================
# Step 6: 評価 (RMSE + Feature Importance) — cat_id 別
# ============================================================
from sklearn.metrics import root_mean_squared_error

# Val 予測 (row_group ごとに読み → cat_id で振り分け → 対応モデルで予測)
pf_val = pq.ParquetFile(VAL_PATH)
all_true = []
all_pred = []
for i in range(pf_val.metadata.num_row_groups):
    rg = pf_val.read_row_group(i).to_pandas()
    rg[FEATURES] = rg[FEATURES].fillna(0)
    pred = np.zeros(len(rg), dtype='float32')
    for cat_id, model in models.items():
        mask = (rg['cat_id'] == cat_id)
        if mask.any():
            pred[mask] = np.clip(model.predict(rg.loc[mask, FEATURES].values), 0, None)
    all_true.append(rg[TARGET].values)
    all_pred.append(pred)
    del rg
del pf_val

val_true = np.concatenate(all_true)
val_pred = np.concatenate(all_pred)
del all_true, all_pred

rmse = root_mean_squared_error(val_true, val_pred)
print(f'Val RMSE (全体): {rmse:.4f}')

# cat_id 別 RMSE
pf_val = pq.ParquetFile(VAL_PATH)
cat_true = {c: [] for c in CAT_NAMES}
cat_pred = {c: [] for c in CAT_NAMES}
for i in range(pf_val.metadata.num_row_groups):
    rg = pf_val.read_row_group(i).to_pandas()
    rg[FEATURES] = rg[FEATURES].fillna(0)
    for cat_id in CAT_NAMES:
        mask = (rg['cat_id'] == cat_id)
        if mask.any():
            cat_true[cat_id].append(rg.loc[mask, TARGET].values)
            cat_pred[cat_id].append(np.clip(models[cat_id].predict(rg.loc[mask, FEATURES].values), 0, None))
    del rg
del pf_val

for cat_id, cat_name in CAT_NAMES.items():
    t = np.concatenate(cat_true[cat_id])
    p = np.concatenate(cat_pred[cat_id])
    print(f'  {cat_name:>10}: RMSE={root_mean_squared_error(t, p):.4f} (n={len(t):,})')
del cat_true, cat_pred, val_true, val_pred

# --- Feature Importance (最初のモデル: FOODS) ---
fi = (
    pd.DataFrame({'feature': FEATURES, 'importance': models[0].feature_importance('gain')})
    .sort_values('importance', ascending=False)
)
fig, ax = plt.subplots(figsize=(10, 8))
fi.head(25).plot.barh(x='feature', y='importance', ax=ax, legend=False)
ax.set_title('Feature Importance — FOODS (Top 25, gain)')
ax.invert_yaxis()
plt.tight_layout()
os.makedirs(DATA_DIR + 'figures', exist_ok=True)
plt.savefig(DATA_DIR + 'figures/feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()
print(fi.head(10).to_string(index=False))

## Step 7: Submission 生成

In [ ]:
# ============================================================
# Step 7: Submission 生成 — cat_id 別モデルで予測 → pivot
# ============================================================

def predict_parquet_to_wide(parquet_path, suffix):
    """parquet を row_group ごとに読み → cat_id 別予測 → pivot"""
    pf = pq.ParquetFile(parquet_path)
    ids, d_nums, preds = [], [], []
    for i in range(pf.metadata.num_row_groups):
        rg = pf.read_row_group(i).to_pandas()
        rg[FEATURES] = rg[FEATURES].fillna(0)
        pred = np.zeros(len(rg), dtype='float32')
        for cat_id, model in models.items():
            mask = (rg['cat_id'] == cat_id)
            if mask.any():
                pred[mask] = np.clip(model.predict(rg.loc[mask, FEATURES].values), 0, None)
        ids.extend(rg['id'].values)
        d_nums.extend(rg['d_num'].values)
        preds.extend(pred)
        del rg
    del pf

    df_pred = pd.DataFrame({'id': ids, 'd_num': d_nums, 'pred': preds})
    del ids, d_nums, preds

    wide = df_pred.pivot(index='id', columns='d_num', values='pred')
    wide.columns = [f'F{i+1}' for i in range(PRED_HORIZON)]
    wide = wide.reset_index()
    wide['id'] = wide['id'].str.replace('_evaluation', f'_{suffix}', regex=False)
    return wide

sub_val  = predict_parquet_to_wide(VAL_PATH,  'validation')
sub_eval = predict_parquet_to_wide(EVAL_PATH, 'evaluation')

submission = pd.concat([sub_val, sub_eval], axis=0).reset_index(drop=True)
del sub_val, sub_eval

print(f'Submission shape: {submission.shape}')
print(f'Expected        : (60980, 29)')
assert submission.shape == (60980, 29), f'Shape mismatch: {submission.shape}'

OUT_SUB = DATA_DIR + 'submission.csv'
submission.to_csv(OUT_SUB, index=False)
print(f'\nSaved: {OUT_SUB}')
print(f'Val RMSE: {rmse:.4f}')
submission.head(3)